## Execise 1

In this exercise, do the following:
1. Load the dataset used in the time series example - Energy consumption data. You can find it in the notebook "TSA_Example" in Time Series folder in Moodle.


In [ ]:
import kagglehub

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

d:\Second semester Data and Data Things\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Download latest version
path = kagglehub.dataset_download("robikscube/hourly-energy-consumption")

print("Path to dataset files:", path)

100%|██████████| 11.4M/11.4M [00:01<00:00, 7.28MB/s]

Extracting files...


Path to dataset files: C:\Users\DELL\.cache\kagglehub\datasets\robikscube\hourly-energy-consumption\versions\3


In [ ]:
import os

file_path = r"C:\Users\DELL\.cache\kagglehub\datasets\robikscube\hourly-energy-consumption\versions\3"

print(os.listdir(file_path))

['AEP_hourly.csv', 'COMED_hourly.csv', 'DAYTON_hourly.csv', 'DEOK_hourly.csv', 'DOM_hourly.csv', 'DUQ_hourly.csv', 'EKPC_hourly.csv', 'est_hourly.paruqet', 'FE_hourly.csv', 'NI_hourly.csv', 'PJME_hourly.csv', 'PJMW_hourly.csv', 'pjm_hourly_est.csv', 'PJM_Load_hourly.csv']


In [ ]:
file_path = r"C:\Users\DELL\.cache\kagglehub\datasets\robikscube\hourly-energy-consumption\versions\3\AEP_hourly.csv"

df = pd.read_csv(file_path)
df.head()

,Datetime,AEP_MW
0,2004-12-31 01:00:00,13478.0
1,2004-12-31 02:00:00,12865.0
2,2004-12-31 03:00:00,12577.0
3,2004-12-31 04:00:00,12517.0
4,2004-12-31 05:00:00,12670.0


In [ ]:
TARGET = "AEP_MW"

In [ ]:
df.columns

Index(['Datetime', 'AEP_MW'], dtype='object')

In [ ]:
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.set_index("Datetime")
df = df.sort_index()
df.head()

,AEP_MW
Datetime,
2004-10-01 01:00:00,12379.0
2004-10-01 02:00:00,11935.0
2004-10-01 03:00:00,11692.0
2004-10-01 04:00:00,11597.0
2004-10-01 05:00:00,11681.0


2. Setup a nested MLFlow loop where different modelling experiments can be tracked and the use the dataset in point 1 to experiment and track models. You should do following combinations:
    1. At least 3 model types
    2. At least 3 different feature combinations
    3. At least 3 different options for 3 different hyperparameters
    4. At least 3 different time splits for train test
3. For each option in the combination, you should calculate & log the following in MLFlow:
    1. RMSE
    2. MAE
    3. Plot of actual vs predicted for 1 month data
    4. Plot of actual vs predicted for 1 week of data
    5. All of the combination info in point 2, such as which model, what feature combindation, what hyperparameter, what train test split has been used
4. Turn on MLFlow UI and track your experiments

In [ ]:
def create_features(df, feature_set):
    data = df.copy()

    if feature_set == "lags":
        data["lag1"] = data[TARGET].shift(1)
        data["lag24"] = data[TARGET].shift(24)
        data["lag168"] = data[TARGET].shift(168)

    elif feature_set == "calendar":
        data["hour"] = data.index.hour
        data["dayofweek"] = data.index.dayofweek
        data["month"] = data.index.month

    elif feature_set == "lags_calendar":
        data["lag1"] = data[TARGET].shift(1)
        data["lag24"] = data[TARGET].shift(24)
        data["hour"] = data.index.hour
        data["dayofweek"] = data.index.dayofweek

    return data.dropna()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

In [ ]:
time_splits = [
    ("2018-01-01", "2018-06-01"),
    ("2018-01-01", "2018-09-01"),
    ("2018-01-01", "2018-12-01"),
]

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

mlflow.set_experiment("Energy_TimeSeries_Experiment")

2026/02/24 21:12:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/24 21:12:27 INFO mlflow.store.db.utils: Updating database tables
2026/02/24 21:12:28 INFO mlflow.tracking.fluent: Experiment with name 'Energy_TimeSeries_Experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///d:/Second semester Data and Data Things/MLOops/mlruns/1', creation_time=1771963948599, experiment_id='1', last_update_time=1771963948599, lifecycle_stage='active', name='Energy_TimeSeries_Experiment', tags={}, workspace='default'>

In [ ]:
feature_sets = ["lags", "calendar", "lags_calendar"]

models = {
    "RandomForest": RandomForestRegressor,
    "XGBoost": XGBRegressor,
    "LinearRegression": LinearRegression
}

rf_params = {"n_estimators": [50, 100, 200]}
xgb_params = {"max_depth": [3, 5, 7]}
lr_params = {"fit_intercept": [True, False]}

with mlflow.start_run(run_name="Master_Run"):

    for split in time_splits:
        for feature_set in feature_sets:
            data = create_features(df, feature_set)

            train = data.loc[:split[1]]
            test = data.loc[split[1]:]

            X_train = train.drop(TARGET, axis=1)
            y_train = train[TARGET]
            X_test = test.drop(TARGET, axis=1)
            y_test = test[TARGET]

            for model_name, model_class in models.items():

                if model_name == "RandomForest":
                    param_grid = rf_params
                elif model_name == "XGBoost":
                    param_grid = xgb_params
                else:
                    param_grid = lr_params

                for param_name, param_values in param_grid.items():
                    for value in param_values:

                        with mlflow.start_run(nested=True):

                            params = {param_name: value}
                            model = model_class(**params)

                            model.fit(X_train, y_train)
                            preds = model.predict(X_test)

                            rmse = np.sqrt(mean_squared_error(y_test, preds))
                            mae = mean_absolute_error(y_test, preds)

                            # Log metrics
                            mlflow.log_metric("RMSE", rmse)
                            mlflow.log_metric("MAE", mae)

                            # Log parameters
                            mlflow.log_param("model", model_name)
                            mlflow.log_param("feature_set", feature_set)
                            mlflow.log_param("train_end", split[1])
                            mlflow.log_param(param_name, value)

                            # ---- Plot 1 Month ----
                            fig1 = plt.figure()
                            y_test[:720].plot(label="Actual")
                            pd.Series(preds[:720], index=y_test.index[:720]).plot(label="Predicted")
                            plt.legend()
                            plt.title("1 Month Prediction")
                            mlflow.log_figure(fig1, "1_month_plot.png")
                            plt.close(fig1)

                            # ---- Plot 1 Week ----
                            fig2 = plt.figure()
                            y_test[:168].plot(label="Actual")
                            pd.Series(preds[:168], index=y_test.index[:168]).plot(label="Predicted")
                            plt.legend()
                            plt.title("1 Week Prediction")
                            mlflow.log_figure(fig2, "1_week_plot.png")
                            plt.close(fig2)

                            mlflow.sklearn.log_model(model, "model")

2026/02/24 21:17:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/24 21:17:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/02/24 21:18:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/24 21:18:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

KeyboardInterrupt: 

In [ ]:
import subprocess
import time

# Start MLFlow UI from the notebook
print("\nStarting MLFlow UI...")
process = subprocess.Popen(["mlflow", "ui", "--port", "5000"])  # Starts MLFlow UI on port 5000

# Wait a moment to ensure the server starts
time.sleep(3)

# Display a link to the MLFlow UI
display.display(display.Markdown("[Open MLFlow UI](http://localhost:5000)"))

print("MLFlow UI is running at http://localhost:5000. Press Ctrl+C in the terminal to stop it.")


Starting MLFlow UI...


AttributeError: 'function' object has no attribute 'display'

## Exercise 1
In this exercise, do the following:
1. Create a function that preprocess new ames data in the same way as the original ames data was preprocessed in step 5 in the `MLOps.ipynb` notebook.


In [1]:
import pandas as pd

def preprocess_new_ames_data(df, training_columns):
    
    # Separate target if present
    if "SalePrice" in df.columns:
        y = df["SalePrice"]
        X = df.drop("SalePrice", axis=1)
    else:
        y = None
        X = df
    
    # One-hot encode categorical variables
    X = pd.get_dummies(X)
    
    # Align columns with training data
    X = X.reindex(columns=training_columns, fill_value=0)
    
    return X, y

2. Create a function that takes as input a new ames dataset and a model. The function should pre-process the new data and evaluate the model on that new data using mean absolute error.


In [2]:
from sklearn.metrics import mean_absolute_error

def evaluate_model_on_new_data(new_data_path, model, training_columns):
    
    df = pd.read_csv(new_data_path)
    
    X_new, y_new = preprocess_new_ames_data(df, training_columns)
    
    predictions = model.predict(X_new)
    
    mae = mean_absolute_error(y_new, predictions)
    
    return mae

3. Test the function from 2. on the "NewAmesData1.csv" dataset and the best model from the `MLOps.ipynb` notebook.


In [6]:
import pandas as pd

ames = pd.read_csv("AmesHousing.csv")

In [7]:
y = ames["SalePrice"]
X = ames.drop("SalePrice", axis=1)

In [8]:
X = pd.get_dummies(X)

In [11]:
training_columns = X_train.columns

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()
model.fit(X_train, y_train)
mae1 = evaluate_model_on_new_data("NewAmesData1.csv", model, training_columns)

print("MAE on NewAmesData1:", mae1)

MAE on NewAmesData1: 96673.81064085447


4. Test the function from 2. on the "NewAmesData2.csv" dataset and the best model from the `MLOps.ipynb` notebook. Do you see any drift?
5. Do you see a data drift in "NewAmesData2.csv"? If so, for which variables?
6. Do you see a data drift in "NewAmesData4.csv"? If so, for which variables?

In [ ]:
import matplotlib.pyplot as plt

def check_drift(train_df, new_df, column):

    plt.hist(train_df[column], alpha=0.5, label="train")
    plt.hist(new_df[column], alpha=0.5, label="new")

    plt.legend()
    plt.title(column)
    plt.show()

: 